# 🎓 Student Success Prediction | توقع نجاح الطالب
**Machine Learning Project — Labs 1, 2 & 3 combined**

By | إعداد: انفال بخيت الدوسري

**Idea | الفكرة:** predict whether a student will **pass (1)** or **fail (0)** using study habits, attendance, and previous score.
نتوقع هل الطالب بينجح أو يرسب بناءً على ساعات المذاكرة ونسبة الحضور ودرجته السابقة.

**Pipeline:** Explore → Split → Feature Engineering → Preprocess → Train → Tune → Evaluate → Explain → Predict

## 1️⃣ Setup & Data | تجهيز البيانات

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
os.makedirs("images", exist_ok=True)

PROJECT_NAME = "Student Success Prediction"
TARGET_COLUMN = "Passed"

# Synthetic dataset (fixed seed => same results every run)
rng = np.random.default_rng(42)
n = 80
study_hours = rng.integers(1, 13, n)            # hours per week
attendance  = rng.integers(50, 101, n)          # % attendance
prev_score  = rng.integers(40, 101, n)          # previous exam score
city        = rng.choice(["Riyadh", "Jeddah", "Dammam"], n)

logit = 0.35*study_hours + 0.05*attendance + 0.04*prev_score - 8.8 + rng.normal(0, 0.8, n)
passed = (1 / (1 + np.exp(-logit)) > 0.5).astype(int)

df = pd.DataFrame({
    "Study_Hours": study_hours,
    "Attendance": attendance,
    "Previous_Score": prev_score,
    "City": city,
    "Passed": passed,
})

# Add a few missing values to make preprocessing meaningful
df.loc[[5, 17, 42], "Attendance"] = np.nan
df.loc[[9, 60], "Previous_Score"] = np.nan

print("Project:", PROJECT_NAME)
print("\nFirst Records:\n", df.head())
print("\nShape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nTarget Distribution:\n", df[TARGET_COLUMN].value_counts())

Project: Student Success Prediction

First Records:
    Study_Hours  Attendance  Previous_Score    City  Passed
0            2        97.0           100.0  Riyadh       1
1           10        72.0            80.0  Jeddah       1
2            8        58.0            64.0  Riyadh       0
3            6        92.0            64.0  Jeddah       0
4            6        82.0            65.0  Jeddah       0

Shape: (80, 5)

Missing Values:
 Study_Hours       0
Attendance        3
Previous_Score    2
City              0
Passed            0
dtype: int64

Target Distribution:
 Passed
1    44
0    36
Name: count, dtype: int64


In [2]:
# Chart 1: target distribution
counts = df[TARGET_COLUMN].value_counts().sort_index()
plt.figure(figsize=(5, 3.5))
plt.bar(["Failed (0)", "Passed (1)"], counts.values, color=["#ef4444", "#22c55e"])
plt.title("Target Distribution")
plt.ylabel("Students")
plt.tight_layout()
plt.savefig("images/target_distribution.png", dpi=150)
plt.show()

## 2️⃣ Feature Engineering | هندسة الخصائص
نسوي خاصية جديدة: **Effort_Index** = ساعات المذاكرة × نسبة الحضور.

In [3]:
df["Effort_Index"] = df["Study_Hours"] * df["Attendance"].fillna(df["Attendance"].median()) / 100
print("🛠️ New Feature: Effort_Index")
print(df.head())
print("Total columns:", df.shape[1])

🛠️ New Feature: Effort_Index
   Study_Hours  Attendance  Previous_Score    City  Passed  Effort_Index
0            2        97.0           100.0  Riyadh       1          1.94
1           10        72.0            80.0  Jeddah       1          7.20
2            8        58.0            64.0  Riyadh       0          4.64
3            6        92.0            64.0  Jeddah       0          5.52
4            6        82.0            65.0  Jeddah       0          4.92
Total columns: 6


## 3️⃣ Train / Test Split | تقسيم البيانات

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))
print("✅ Data split successfully!")

Training Records: 60
Testing Records: 20
✅ Data split successfully!


## 4️⃣ Preprocessing Pipeline | خط المعالجة
أرقام: Imputer (median) + Scaler. نصوص (City): Imputer + OneHot.

In [5]:
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
     make_column_selector(dtype_include="number")),
    (make_pipeline(SimpleImputer(strategy="most_frequent"),
                   OneHotEncoder(handle_unknown="ignore")),
     make_column_selector(dtype_exclude="number")),
)
print("🔧 Preprocessing pipeline ready!")

🔧 Preprocessing pipeline ready!


## 5️⃣ Train Models | تدريب النماذج
نقارن 3 نماذج: Logistic Regression, Random Forest, Gradient Boosting.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": make_pipeline(prep, LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipeline(prep, RandomForestClassifier(random_state=42)),
    "Gradient Boosting": make_pipeline(prep, GradientBoostingClassifier(random_state=42)),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    print(f"✅ {name} trained!")

✅ Logistic Regression trained!
✅ Random Forest trained!
✅ Gradient Boosting trained!


## 6️⃣ Evaluate | التقييم

In [7]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix)

results = []
for name, m in models.items():
    pred = m.predict(X_test)
    prob = m.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred), 2),
        "F1": round(f1_score(y_test, pred, zero_division=0), 2),
        "ROC-AUC": round(roc_auc_score(y_test, prob), 2),
        "PR-AUC": round(average_precision_score(y_test, prob), 2),
    })
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

              Model  Accuracy   F1  ROC-AUC  PR-AUC
Logistic Regression      0.85 0.87     0.96    0.97
      Random Forest      0.80 0.83     0.86    0.90
  Gradient Boosting      0.70 0.75     0.84    0.88


In [8]:
# Chart 2: model comparison
results_df.set_index("Model")[["Accuracy", "F1", "ROC-AUC", "PR-AUC"]].plot(
    kind="bar", figsize=(8, 4), ylim=(0, 1.05), rot=15)
plt.title("Model Comparison")
plt.tight_layout()
plt.savefig("images/model_comparison.png", dpi=150)
plt.show()

## 7️⃣ Hyperparameter Tuning | تحسين الإعدادات

In [9]:
from sklearn.model_selection import GridSearchCV

params = {
    "gradientboostingclassifier__n_estimators": [20, 50],
    "gradientboostingclassifier__learning_rate": [0.05, 0.1],
    "gradientboostingclassifier__max_depth": [1, 2],
}
search = GridSearchCV(
    make_pipeline(prep, GradientBoostingClassifier(random_state=42)),
    params, scoring="average_precision", cv=3
)
search.fit(X_train, y_train)
best_model = search.best_estimator_

print("🔧 Best Parameters:", search.best_params_)
print("📊 Best CV PR-AUC:", round(search.best_score_, 2))

🔧 Best Parameters: {'gradientboostingclassifier__learning_rate': 0.05, 'gradientboostingclassifier__max_depth': 1, 'gradientboostingclassifier__n_estimators': 50}
📊 Best CV PR-AUC: 0.87


## 8️⃣ Final Evaluation & Threshold | التقييم النهائي وحد القرار

In [10]:
probabilities = best_model.predict_proba(X_test)[:, 1]
threshold = 0.5
predictions = (probabilities >= threshold).astype(int)

print("🎯 Threshold:", threshold)
print("📊 PR-AUC:", round(average_precision_score(y_test, probabilities), 2))
print("📈 F1 Score:", round(f1_score(y_test, predictions, zero_division=0), 2))
cm = confusion_matrix(y_test, predictions)
print("🔍 Confusion Matrix:\n", cm)

# Chart 3: confusion matrix
plt.figure(figsize=(4, 3.5))
plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
plt.xticks([0, 1], ["Fail", "Pass"]); plt.yticks([0, 1], ["Fail", "Pass"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("images/confusion_matrix.png", dpi=150)
plt.show()

🎯 Threshold: 0.5
📊 PR-AUC: 0.93
📈 F1 Score: 0.8
🔍 Confusion Matrix:
 [[ 5  4]
 [ 1 10]]


## 9️⃣ Explain the Model | تفسير النموذج

In [11]:
names = best_model[:-1].get_feature_names_out()
names = [n.split("__")[-1] for n in names]
importance = pd.Series(best_model[-1].feature_importances_, index=names).sort_values(ascending=False)

print("📊 Feature Importance:")
print(importance.round(2))
print("🏆 Most Important Feature:", importance.index[0])

# Chart 4: feature importance
plt.figure(figsize=(6, 4))
importance.sort_values().plot(kind="barh", color="#6366f1")
plt.title("Feature Importance")
plt.tight_layout()
plt.savefig("images/feature_importance.png", dpi=150)
plt.show()

📊 Feature Importance:
Effort_Index      0.84
Previous_Score    0.11
City_Jeddah       0.04
Attendance        0.01
Study_Hours       0.00
City_Dammam       0.00
City_Riyadh       0.00
dtype: float64
🏆 Most Important Feature: Effort_Index


## 🔮 Try It: Predict a New Student | جرّب طالب جديد

In [12]:
new_student = pd.DataFrame([{
    "Study_Hours": 9,
    "Attendance": 90,
    "Previous_Score": 78,
    "City": "Riyadh",
    "Effort_Index": 9 * 90 / 100,
}])
prob = best_model.predict_proba(new_student)[0, 1]
print(f"Probability of passing: {prob:.0%}")
print("Prediction:", "✅ Pass" if prob >= threshold else "❌ Fail")

Probability of passing: 89%
Prediction: ✅ Pass


## ✅ Conclusion | الخلاصة
- Built a complete ML pipeline: preprocessing → feature engineering → model comparison → tuning → explanation.
- Logistic Regression scored best on the small test set, while tuned Gradient Boosting was used for feature importance and the final demo. With only 80 rows, results vary a lot between models.
- **Note:** the dataset is small & synthetic, so results are for learning purposes only.

بنينا مشروع تعلم آلي كامل، والبيانات صغيرة ومولّدة للتعلم فقط.